# FIT4112 · LAB 4 · Notebook 01
## Chuẩn bị dữ liệu và EDA chuỗi thời gian

**Mục tiêu:** chọn một trạm, kiểm tra khoảng trống theo giờ, xử lý dữ liệu cảm biến lỗi và trực quan hóa theo thời gian. **Dữ liệu mặc định là GIẢ LẬP**, không phản ánh thực tế Bắc Kinh.

**Câu hỏi:** Vì sao dữ liệu theo giờ không được tùy tiện chia ngẫu nhiên hoặc điền giá trị thiếu bằng trung bình của cả chuỗi?

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab4.py").exists()), None)
assert ROOT is not None, "Mở notebook trong thư mục repo fit4112_lab_4."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.lab4 import load_raw, prepare_hourly

# ===== PARAMETERS =====
DATA_SOURCE = "demo"      # "demo" (không cần tải file) hoặc "uci_zip" (nâng cao)
STATION = "Aotizhongxin"  # chỉ dùng khi DATA_SOURCE="uci_zip"

raw = load_raw(ROOT, source=DATA_SOURCE, station=STATION)
print("Số bản ghi thô:", len(raw))
display(raw.head())

### 1. Chất lượng dữ liệu

Đếm số giá trị trống/âm, sau đó ghép dữ liệu về lưới thời gian đúng 1 giờ. Hàm xử lý **chỉ forward-fill theo quá khứ** để tạo lag; đồng thời giữ cờ observed_target nhằm tránh lấy giá trị tự điền làm nhãn thật.

In [ ]:
raw_pm = pd.to_numeric(raw["PM2.5"], errors="coerce")
print("PM2.5 trống:", int(raw_pm.isna().sum()))
print("PM2.5 âm (không hợp lệ):", int((raw_pm < 0).sum()))
hourly = prepare_hourly(raw)
print("Số mốc giờ sau chuẩn hóa:", len(hourly))
print("Số giờ không có nhãn đo hợp lệ:", int((~hourly["observed_target"]).sum()))
display(hourly.head(8))

out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
hourly.to_csv(out / "hourly_pm25.csv", index=False)
print("Saved:", out / "hourly_pm25.csv")

### 2. Trực quan hóa

Quan sát mức độ thay đổi theo giờ, trung bình trượt **chỉ nhìn quá khứ** và chu kỳ 24 giờ. Không dùng đồ thị toàn bộ data để chọn hyperparameter dựa trên test.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly["datetime"], hourly["pm25"], label="PM2.5 (ffill history)", linewidth=1)
ax.plot(hourly["datetime"], hourly["pm25"].rolling(24, min_periods=1).mean(),
        label="Rolling mean 24h", linewidth=2)
ax.set(title="Hourly PM2.5 — demo / selected station", xlabel="Time", ylabel="PM2.5")
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

profile = hourly.groupby(hourly["datetime"].dt.hour)["pm25"].mean()
display(profile.rename("Mean PM2.5 by hour").to_frame())
profile.plot(figsize=(8, 3), marker="o", title="Hourly pattern (descriptive only)")
plt.xlabel("Hour of day")
plt.tight_layout()
plt.show()

### Trả lời trong báo cáo
1. Một hàng của hourly_pm25.csv tương ứng với điều gì?
2. Có bao nhiêu giờ không có giá trị đo hợp lệ? Giá trị ấy có được dùng làm nhãn đánh giá không?
3. Vì sao forward-fill phù hợp hơn backfill khi mô phỏng dự báo từng giờ?
4. Dữ liệu demo có một đoạn PM2.5 tăng; có thể kết luận ô nhiễm Bắc Kinh tăng thật không? Vì sao?